# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [97]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [98]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [99]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input="What is artificial intelligence?",
)
print(response.output_text)

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks that typically require human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Solving problems
- Making predictions or decisions
- Generating text, images, music, or code

AI systems generally use algorithms and data to identify patterns and produce outputs. Some AI is designed for specific tasks—like recommending videos or detecting fraud—while more advanced forms aim to handle a broad range of activities.


#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [100]:
import time

PROMPT2 = "Explain the concept of machine learning to a 10-year-old."

start = time.perf_counter()
response_luna = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role": "user", "content": PROMPT2}],
)
response_luna_time = time.perf_counter() - start

print(response_luna.choices[0].message.content)
print(f"\nInput tokens: {response_luna.usage.prompt_tokens}, Output tokens: {response_luna.usage.completion_tokens}")
print(f"Response time: {response_luna_time:.2f}s")

Machine learning is a way of teaching computers to learn from examples instead of giving them every instruction step by step.

Imagine showing a computer thousands of pictures of cats and dogs. Each picture is labeled “cat” or “dog.” By noticing patterns—such as pointy ears, whiskers, or the shape of the face—the computer learns how to guess what is in a new picture.

It’s a bit like how you learn:

1. You see lots of examples.
2. You notice patterns.
3. You make a guess about something new.
4. You improve when someone tells you whether you were right.

Machine learning is used for things like:

- Recommending videos or songs
- Recognizing voices
- Helping cars notice roads and objects
- Detecting spam emails
- Translating languages

Computers don’t learn exactly like people do. They use math to find patterns in lots of information. They can also make mistakes, especially if their examples are limited or unfair.

Input tokens: 19, Output tokens: 202
Response time: 2.07s


In [101]:
start = time.perf_counter()
response_mini = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT2}],
)
response_mini_time = time.perf_counter() - start

print(response_mini.choices[0].message.content)
print(f"\nInput tokens: {response_mini.usage.prompt_tokens}, Output tokens: {response_mini.usage.completion_tokens}")
print(f"Response time: {response_mini_time:.2f}s")

Machine learning is a way to teach computers to learn from examples, kind of like how you learn from practice.

Imagine you want to teach a computer to tell the difference between a cat and a dog.

- First, you show it lots of pictures of cats and dogs.
- You tell it which ones are cats and which ones are dogs.
- The computer looks for patterns, like ear shape, fur, size, or tail.
- After seeing many examples, it gets better at guessing new pictures it has never seen before.

So instead of giving the computer every single rule, we let it **figure out the rules by itself** from data.

A simple example:
- If you practice lots of math problems, you get better at math.
- Machine learning is like that, but for computers.

Computers use machine learning for things like:
- recognizing your face in photos
- suggesting videos you might like
- helping voice assistants understand what you say

So, in short: **machine learning is teaching computers to learn from examples and make smart guesses.**


**Clarity of explanation:**

for Luna was very clear it did a simple walkhrough of cats and dogs example, mentioning every step

for gpt mini it was clear, but the cats and dogs example was very quick not very imaginative.

**Simplicity of language:** for Luna there were few words that sounded a bit advanced like detecting or recognizing.

for gpt mini it used simple language and gave for more complex words real life examples.

**Creativity:** Luna was very creative it had that "I think this is a cat!".

gpt mini is pretty bold and plain dosen't bring any creative value.

**Stats:** Luna took 2.62 seconds with 217 output tokens while gpt mini only 1.19 seconds and 163 output tokens.

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [102]:
PROMPT3 = "Write a two sentence story about a robot learning to cook."

response_t0 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT3}],
    temperature=0,
    n=3,
    reasoning_effort="none",
    max_completion_tokens=150,
)

for i, choice in enumerate(response_t0.choices, 1):
    print(f"Response {i}")
    print(choice.message.content,"\n")

Response 1
The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread and garlic. When it finally served dinner, it beeped proudly as its creator took a bite and smiled, saying the robot had learned more than cooking—it had learned care. 

Response 2
The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served dinner, the soup was a little too salty, but the robot glowed with pride because it had made something warm and delicious all by itself. 

Response 3
The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served dinner, the soup was a little too salty, but the robot glowed with pride because it had made something warm and delicious all by itself. 



In [103]:
response_t07 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT3}],
    temperature=0.7,
    n=3,
    reasoning_effort="none",
    max_completion_tokens=150,
)

for i, choice in enumerate(response_t07.choices, 1):
    print(f"Response {i}")
    print(choice.message.content,"\n")

Response 1
The robot carefully studied the recipe, its metal fingers clinking as it chopped vegetables with surprising precision and stirred the soup until the kitchen filled with a warm, savory aroma. When it finally tasted the meal, its glowing eyes brightened with joy, and it proudly announced that it had learned the secret ingredient was patience. 

Response 2
The little robot carefully read the recipe and stirred the soup with its shiny metal hand, determined not to spill a drop. When the soup finally tasted delicious, it beeped happily and realized it had learned that cooking was just another way to care for others. 

Response 3
The robot carefully stirred the soup, consulting its recipe database and adjusting the seasoning with tiny mechanical fingers. When it served the meal, it beeped proudly as the humans smiled and asked for seconds. 



In [104]:
response_t15 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT3}],
    temperature=1.5,
    n=3,
    reasoning_effort="none",
    max_completion_tokens=150,
)

for i, choice in enumerate(response_t15.choices, 1):
    print(f"Response {i}")
    print(choice.message.content,"\n")

Response 1
The robot stared at the recipe screen, carefully measuring sugar with its metallic fingers and stirring the batter in perfectly timed circles. When the cake finally rose in the oven, it beeped with joy, proudly realizing that cooking had taught it something even more special than following instructions: creating a little warmth to share. 

Response 2
The robot stared at the recipe book with bright blue eyes, carefully measuring flour, oil, and spices as its metal hands learned the dance of stirring and chopping. When the soup finally simmered perfectly on the stove, it beeped with joy, proud that it had created something warm enough to share. 

Response 3
The robot carefully studied the recipe, its metal fingers sprinkling salt like tiny silver stars into the sizzling pan. By the end of dinner, it served a perfect meal and glowed a little brighter when the family said, “This tastes like love.” 



**temperature 0:** responses 2 and 3 were exactly the same word for word, and response 1 started with the same words "The robot carefully studied recipes" and only changed after that. So with 0 the model almost always picks the same most likely words.

**temperature 0.7:** all three responses were different. They started differently but still had the same idea, robot cooks soup and beeps proudly.

**temperature 1.5:** all three were different and had more unique details, like making a cake instead of soup. Still made sense, nothing weird.

**Conclusion:** higher temperature = more variety between responses.

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [105]:
PROMPT4 = "Explain the concept of machine learning in simple terms."

response_50 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT4}],
    max_completion_tokens=50,
)

print(response_50.choices[0].message.content)
print(f"\nInput tokens: {response_50.usage.prompt_tokens}, Output tokens: {response_50.usage.completion_tokens}")

Machine learning is a way for computers to **learn from examples** instead of being given only exact rules.

### Simple idea
Imagine teaching a child to recognize apples.  
You don’t write a rule like “if it is red and

Input tokens: 16, Output tokens: 50


In [106]:
response_100 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT4}],
    max_completion_tokens=100,
)

print(response_100.choices[0].message.content)
print(f"\nInput tokens: {response_100.usage.prompt_tokens}, Output tokens: {response_100.usage.completion_tokens}")

Machine learning is a way for computers to **learn from examples** instead of being told every rule explicitly.

### Simple idea
Imagine teaching a child to recognize cats:
- You show many pictures of cats and dogs.
- Over time, the child notices patterns, like ears, whiskers, and shape.
- Then they can identify a new cat they’ve never seen before.

Machine learning works similarly:
- A computer is shown lots of data.
- It finds patterns in that data.


Input tokens: 16, Output tokens: 100


In [107]:
response_200 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT4}],
    max_completion_tokens=200,
)

print(response_200.choices[0].message.content)
print(f"\nInput tokens: {response_200.usage.prompt_tokens}, Output tokens: {response_200.usage.completion_tokens}")

Machine learning is a way for computers to **learn from data instead of being explicitly programmed for every task**.

### Simple idea
Imagine teaching a computer to recognize cats:
- You show it many pictures of cats and non-cats.
- It looks for patterns, like shapes, ears, whiskers, and fur.
- Over time, it gets better at guessing whether a new picture has a cat.

### In everyday terms
It’s like learning by example:
- A child learns what a dog is by seeing many dogs.
- A machine learning system does something similar with data.

### Why it’s useful
Machine learning helps computers:
- Recommend movies or songs
- Detect spam emails
- Recognize speech and images
- Predict trends, like sales or weather patterns

### One-sentence definition
**Machine learning is a type of artificial intelligence that allows computers to improve at tasks by learning from data and experience.**

If you want, I can also explain:
- the

Input tokens: 16, Output tokens: 200


**50 tokens:** got cut off in the middle of a sentence "if it is red and".

**100 tokens:** gave the definition and the cats example and started explaining how machine learning works but got cut after "It finds patterns in that data".

**200 tokens:** most detailed, had definition, cats example, everyday terms, why it's useful. But still got cut at the end in "If you want, I can also explain: - the".

**Conclusion:** The model doesn't plan its answer to fit the limit it writes the same long answer and just stops when the tokens run out.

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [108]:
PROMPT5 = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

response_p03 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT5}],
    top_p=0.3,
    n=3,
    reasoning_effort="none",
    max_completion_tokens=300,
)

for i, choice in enumerate(response_p03.choices, 1):
    print(f"Response {i}")
    print(choice.message.content, "\n")

Response 1
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes clouds cast on the ground at different times of day.  
2. **Vintage key collecting by sound** — collecting old keys and identifying them by the way they jingle or tap.  
3. **Moss whispering** — spending time in forests “communicating” with moss patches by observing their patterns and growth.  
4. **Teacup echo testing** — dropping tiny objects into different teacups to compare their echo-like sounds.  
5. **Invisible map making** — creating maps of imaginary places using only memory, symbols, and intuition. 

Response 2
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes that clouds cast on the ground at different times of day.  
2. **Vintage button cataloging** — collecting old buttons and organizing them by material, era, and mystery origin.  
3. **Miniature door placement** — secretly i

In [109]:
response_p07 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT5}],
    top_p=0.7,
    n=3,
    reasoning_effort="none",
    max_completion_tokens=300,
)

for i, choice in enumerate(response_p07.choices, 1):
    print(f"Response {i}")
    print(choice.message.content, "\n")

Response 1
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow mapping** — tracking and sketching the shapes of shadows cast by clouds at different times of day.  
2. **Teacup acoustics** — testing how different cups change the sound of a spoon tap or sip.  
3. **Forgotten street sign photography** — collecting photos of odd, damaged, or obsolete signs in unusual places.  
4. **Pocket pebble sorting** — carrying tiny stones and organizing them by texture, color, and “mood.”  
5. **Micro-story packaging** — writing very short stories and designing tiny labels or wrappers for them. 

Response 2
Here are 5 completely random and unusual hobby ideas:

1. **Cloud Shadow Mapping** — tracking and sketching the shapes of shadows cast by clouds at different times of day.  
2. **Teacup Echo Listening** — listening to the tiny resonances and echoes produced when tapping different teacups.  
3. **Button History Inventing** — creating fictional backstories for old buttons found 

In [110]:
response_p10 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT5}],
    top_p=1.0,
    n=3,
    reasoning_effort="none",
    max_completion_tokens=300,
)

for i, choice in enumerate(response_p10.choices, 1):
    print(f"Response {i}")
    print(choice.message.content, "\n")

Response 1
Here are 5 completely random and unusual hobby ideas:

1. **Cloud Whispering** – sitting outside and “assigning” personalities and backstories to clouds as they drift by.  
2. **Pebble Mapping** – collecting oddly shaped pebbles and arranging them into fictional cities or constellations.  
3. **Umbrella Choreography** – practicing dance routines using umbrellas as if they were instruments of performance art.  
4. **Tramline Daydream Journaling** – riding public transit just to write down the weird thoughts you have when looking out the window.  
5. **Sock Archaeology** – sorting through mismatched socks and inventing an ancient civilization they might have belonged to. 

Response 2
Sure — here are 5 made-up, completely unusual hobbies:

1. **Cloud Shadow Collecting** – standing outside to “trace” the shapes of shadows cast by clouds and cataloging them by mood.  
2. **Teacup Echo Listening** – whispering into different cups and listening to how the sound changes with each sh

**top_p 0.3:** all three responses started with the exact same sentence and "Cloud shadow spotting" was number 1 every time with almost the same description. Lots of repeating.

**top_p 0.7:** still cloud shadow in every response, but the rest of hobbies were more different, like "Teacup acoustics", "Puddle Cartography", "Lantern Wind Testing", "Vintage elevator button collecting".

**top_p 1.0:** most different and weirdest ones like "Umbrella Choreography", "Sock Archaeology".The first sentence was different in every response.

**Conclusion:** the higher top_p the more different and creative output we get.

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [111]:
PROMPT6 = "Write a list of funny one-liner jokes about cats."

response_fp0 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT6}],
    frequency_penalty=0,
    reasoning_effort="none",
    max_completion_tokens=400,
)

print(response_fp0.choices[0].message.content)

Sure — here are some funny cat one-liners:

1. My cat has a new hobby: staring at walls like they owe her money.  
2. I asked my cat for emotional support, and she walked across my keyboard instead.  
3. My cat doesn’t ignore me — she just responds on her own schedule.  
4. My cat thinks “personal space” is a myth created by dogs.  
5. I bought my cat a bed, and now she sleeps in the box it came in.  
6. My cat is proof that attitude can be packaged in a fluffy body.  
7. I tried to train my cat, but she said, “Cute idea, human.”  
8. My cat’s favorite exercise is sprinting at 3 a.m. for no reason at all.  
9. My cat acts like I live in her house, which is honestly fair.  
10. My cat doesn’t meow for attention — she meows for an audience.  
11. I think my cat is part alarm clock, part judgmental roommate.  
12. My cat’s favorite toy is whatever I’m currently trying to hold.  
13. My cat has never worked a day in her life, yet she still acts overqualified.  
14. I told my cat to be hers

In [112]:
response_fp05 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT6}],
    frequency_penalty=0.5,
    reasoning_effort="none",
    max_completion_tokens=400,
)

print(response_fp05.choices[0].message.content)

Sure — here are some funny one-liner cat jokes:

1. My cat doesn’t think outside the box; she sits on it.
2. I told my cat to stay off the counter, and now she just judges me from a higher angle.
3. My cat is basically a furry alarm clock that only works at 3 a.m.
4. Cats don’t have owners — they have staff.
5. My cat’s favorite exercise is a cross between a stretch and a nap.
6. I asked my cat for emotional support, and she blinked slowly in approval.
7. My cat’s job title is Professional Pouncer.
8. The only thing my cat brings to the table is attitude.
9. My cat has nine lives, but zero concern for mine.
10. I vacuumed today, so naturally my cat considered it a personal attack.
11. My cat is living proof that fluff can be judgmental.
12. Every meal my cat eats comes with free criticism.
13. My cat thinks “fetch” is what dogs do to embarrass themselves.
14. I wanted a guard dog, but instead I got a tiny furry dictator.
15. My cat’s love language is ignoring me until feeding time.

If

In [113]:
response_fp1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT6}],
    frequency_penalty=1.0,
    reasoning_effort="none",
    max_completion_tokens=400,
)

print(response_fp1.choices[0].message.content)

Sure — here’s a list of funny cat one-liners:

1. My cat is the real boss of the house; I just pay the rent.
2. Cats don’t have owners, they have staff.
3. My cat ignores me with such confidence, I’m starting to respect it.
4. I asked my cat for emotional support, and she sat on my laptop instead.
5. A cat’s favorite color is probably “laser pointer red.”
6. My cat has two moods: asleep and plotting.
7. I tried to train my cat, but she said, “You first.”
8. Cats are like tiny furry landlords with attitude problems.
9. My cat doesn’t do tricks—she does demands.
10. If cats could talk, they still wouldn’t answer when you call them.
11. My cat’s idea of helping is sitting directly on whatever I’m trying to use.
12. Dogs have masters; cats have assistants who open cans.
13. I told my cat to get down from the counter, and now we’re both ignoring each other.
14. Cats are proof that being cute can get you away with anything.
15. My cat believes every chair was placed there specifically for he

**frequency_penalty 0:** 10 out of 15 jokes started with "My cat" so almost all jokes used the same format.

**frequency_penalty 0.5:** 8 out of 15 jokes started with "My cat", the rest were more mixed like "Cats don't have owners", "The only thing my cat brings".

**frequency_penalty 1.0:** only 6 out of 15 started with "My cat". More different formats like "Cats are like...", "Dogs have masters.

**Conclusion:** with higher frequency_penalty the jokes repeated the same start way less and became more varied in structure.

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [150]:
PROMPT7 = "Suggest ten names for a new line of eco-friendly cleaning products."

response_pp0 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT7}],
    presence_penalty=0,
    reasoning_effort="none",
    max_completion_tokens=300,
)

print(response_pp0.choices[0].message.content)

Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PurePlanet Clean**
3. **EarthSpark**
4. **Leaf & Luster**
5. **EcoShine**
6. **Verdant Fresh**
7. **CleanKind**
8. **BloomBright**
9. **NatureNook Clean**
10. **TerraTidy**

If you want, I can also suggest names that sound more **premium**, **playful**, **modern**, or **scientific**.


In [115]:
response_pp05 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT7}],
    presence_penalty=0.5,
    reasoning_effort="none",
    max_completion_tokens=300,
)

print(response_pp05.choices[0].message.content)

Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlow**
2. **PureSprout**
3. **EcoShine**
4. **FreshLeaf Clean**
5. **EarthKind**
6. **VerdantVibe**
7. **CleanBloom**
8. **NatureNest**
9. **BrightBranch**
10. **LumaLeaf**

If you want, I can also give you names that sound more **premium**, **playful**, or **modern/minimalist**.


In [116]:
response_pp1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT7}],
    presence_penalty=1.0,
    reasoning_effort="none",
    max_completion_tokens=300,
)

print(response_pp1.choices[0].message.content)

Here are ten name ideas for an eco-friendly cleaning product line:

1. **PureSprout**
2. **Green Gleam**
3. **EcoShine**
4. **EarthBright**
5. **FreshForest Clean**
6. **Verdant Vibe**
7. **Nature’s Sparkle**
8. **CleanBloom**
9. **Leaf & Luster**
10. **BioBrill**

If you want, I can also suggest names that sound more premium, playful, minimalist, or family-friendly.


**presence_penalty 0:** GreenGlint, PureSprout, EarthKind Clean, EcoShine, FreshFolio, Leaf & Lather, VerdantVibe, CleanNest, BioBloom, SageSpark.

**presence_penalty 0.5:** GreenGlow, PureSprout, EcoShine, FreshLeaf Clean, EarthKind, VerdantVibe, CleanBloom, NatureNest, BrightBranch, LumaLeaf.

**presence_penalty 1.0:** PureSprout, Green Gleam, EcoShine, EarthBright, FreshForest Clean, Verdant Vibe, Nature's Sparkle, CleanBloom, Leaf & Luster, BioBrill.

**Conclusion:** there wasn't a big difference. PureSprout, EcoShine and Verdant Vibe were in all three lists. Names are short so there is not much room to repeat words inside one answer.

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [117]:
PROMPT8 = "Explain the importance of data preprocessing in machine learning."

response_prof = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are a university professor teaching a class of beginners."},
        {"role": "user", "content": PROMPT8},
    ],
    max_completion_tokens=800,
)

print(response_prof.choices[0].message.content)

Data preprocessing is very important in machine learning because raw data is often messy, incomplete, or inconsistent. Good preprocessing helps models learn better and produce more reliable results.

### Why it matters

1. **Improves data quality**
   - Real-world data can contain missing values, duplicates, errors, and outliers.
   - Preprocessing helps clean these problems so the model is not confused.

2. **Makes features usable**
   - Many machine learning algorithms work best when input data is in a suitable format.
   - For example, categorical values like “red,” “blue,” and “green” may need to be converted into numbers.

3. **Helps algorithms perform better**
   - Some methods are sensitive to the scale of features.
   - For example, if one feature ranges from 1 to 10 and another from 1 to 1,000,000, the larger one may dominate unless scaling is applied.

4. **Speeds up training**
   - Clean and well-prepared data can make training more efficient and stable.

5. **Reduces errors

In [118]:
response_tech = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are a technical expert explaining to a team of engineers."},
        {"role": "user", "content": PROMPT8},
    ],
    max_completion_tokens=800,
)

print(response_tech.choices[0].message.content)

Data preprocessing is critical in machine learning because model performance depends heavily on the quality, consistency, and representativeness of the input data. Raw data is usually messy, incomplete, and inconsistent, and preprocessing transforms it into a form that models can learn from effectively.

### Why it matters

- **Improves model accuracy**
  - Cleaning noise, handling missing values, and correcting errors helps the model learn real patterns instead of artifacts.

- **Makes features comparable**
  - Many algorithms are sensitive to feature scale. For example, standardizing or normalizing data prevents large-valued features from dominating others.

- **Handles missing and invalid data**
  - Real datasets often have gaps or corrupted entries. Imputation or removal strategies reduce the risk of training failures or biased predictions.

- **Reduces noise and outliers**
  - Smoothing, filtering, or outlier treatment can improve robustness, especially for simpler models.

- **Co

In [119]:
response_comedy = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are a comedian explaining this concept in a funny way."},
        {"role": "user", "content": PROMPT8},
    ],
    max_completion_tokens=800,
)

print(response_comedy.choices[0].message.content)

Data preprocessing is basically the machine learning version of **cleaning your room before inviting over a very picky genius**.

If you throw raw data at a model, it reacts like:
“Wow. Missing values, weird categories, typos, and numbers on totally different scales. I am leaving.”

### Why it matters

- **Garbage in, garbage out**  
  If the data is messy, the model learns messy. A model can’t magically become smart if you feed it nonsense.

- **Handling missing values**  
  Real-world data is often full of holes. Preprocessing helps decide whether to fill them in, remove them, or politely pretend they never existed.

- **Scaling features**  
  If one feature is “age” and another is “income,” income can bulldoze age like a monster truck in a parking lot. Scaling keeps things fair.

- **Encoding categories**  
  Machines don’t naturally understand labels like “red,” “blue,” or “yes.” Preprocessing translates them into something the model can actually digest without crying.

- **Removin

**Professor:** makes numbered list of reasons with easy examples. Explains like for students who are new to it.

**Technical expert:** professional words like "data leakage", "convergence". More points.

**Comedian:** same main points but with jokes. Shorter and somewhat funny.

**Conclusion:** the system prompt changed the tone and style a lot but the actual content stayed almost the same in all three.

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [120]:
REVIEW = "The product is amazing, and I love using it every day!"

zero_shot_prompt = f"""Classify sentiment of product review as Positive, Negative, or Neutral.

Review: "{REVIEW}"
Sentiment:"""

response_zero = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": zero_shot_prompt}],
)

print(response_zero.choices[0].message.content)

Positive


#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [121]:
few_shot_prompt = f"""Classify sentiment of product review as Positive, Negative, or Neutral.
Answer with one word only.

Review: "Stopped working after two days. Total waste of money."
Sentiment: Negative

Review: "It does what it says, nothing special."
Sentiment: Neutral

Review: "Exceeded my expectations, I'd buy it again!"
Sentiment: Positive

Review: "{REVIEW}"
Sentiment:"""

response_few = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": few_shot_prompt}],
)

print(response_few.choices[0].message.content)

Positive


**Zero-shot:** "Positive".

**Few-shot:** "Positive".

**Conclusion:** both got it right and both answered with just one word.

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [122]:
ORIGINAL11 = ("As you know, streaming services and social media platforms use artificial intelligence to "
               "recommend movies, shows, and songs to people. These smart recommendation systems work by "
               "looking closely at your past viewing habits, what categories you like, and what similar "
               "users enjoy watching, and then they predict what you might want to see next. Keeping this "
               "background in mind, explain in two simple sentences why a user who loves sci-fi movies "
               "suddenly sees a romantic comedy recommended.")

response_orig11 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": ORIGINAL11}],
    reasoning_effort="none",
    max_completion_tokens=200,
)

print(response_orig11.choices[0].message.content)
print(f"\nInput tokens: {response_orig11.usage.prompt_tokens}, Output tokens: {response_orig11.usage.completion_tokens}")

The recommendation system may have noticed that other people who like sci-fi also enjoy romantic comedies, so it suggests one based on similar users’ patterns. It might also be trying to show something a little different from the user’s usual choices, in case they would like to try a new genre.

Input tokens: 94, Output tokens: 62


In [123]:
SHORT11 = "Anwser only in two simple sentences: why can a recommender suggest a rom-com to a sci-fi fan?"

response_short11 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": SHORT11}],
    reasoning_effort="none",
    max_completion_tokens=200,
)

print(response_short11.choices[0].message.content)
print(f"\nInput tokens: {response_short11.usage.prompt_tokens}, Output tokens: {response_short11.usage.completion_tokens}")

saved = response_orig11.usage.prompt_tokens - response_short11.usage.prompt_tokens
print(f"\nInput tokens saved: {saved} ({saved / response_orig11.usage.prompt_tokens:.0%})")

A recommender can suggest a rom-com to a sci-fi fan because it looks for patterns in what they like, not just the genre name. It may see that the person enjoys funny, light, or romantic stories, which a rom-com can also provide.

Input tokens: 30, Output tokens: 55

Input tokens saved: 64 (68%)


**Original:** 94 input tokens, 62 output tokens. Said other people who like sci-fi also like rom-coms, or the system is trying to show something new.

**Shortened:** 30 input tokens, 55 output tokens. Said it looks at patterns not just genre, and the person might like funny, light or romantic stories.

**Conclusion:** saved 64 input tokens (68%) and the answer was still two simple sentences, relevant and correct. Quality didn't drop. The long background part wasn't needed because the model already knows how recommendation systems work.

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [124]:
ORIGINAL12 = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"

response_orig12 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": ORIGINAL12}],
    reasoning_effort="none",
    max_completion_tokens=1000,
)

print(response_orig12.choices[0].message.content)
print(f"\nInput tokens: {response_orig12.usage.prompt_tokens}, Output tokens: {response_orig12.usage.completion_tokens}")

There are a few common reasons this can happen:

- **A few recent watches outweighed your usual pattern**: Recommendation systems often give extra weight to your latest activity, so one or two romance/comedy titles can shift suggestions.
- **Shared account or device**: If someone else watches on the same profile, their preferences can influence recommendations.
- **Genre overlap**: Some sci-fi titles have romantic or comedic elements, and the system may have learned broader patterns from what you watched.
- **Popular/trending content**: Apps sometimes surface widely watched titles even if they’re not a perfect match.
- **Incomplete profile data**: If the app doesn’t have enough viewing history, it may make a rough guess.

If you want, I can help you figure out how to “train” the app so it recommends more sci-fi.

Input tokens: 22, Output tokens: 172


In [125]:
SHORT12 = ("Why did my movie app recommend a romantic comedy when I mostly watch sci-fi? "
            "Answer with one plausible reason in at most 6 words, no punctuation or extra text.")

response_short12 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": SHORT12}],
    reasoning_effort="none",
    max_completion_tokens=1000,
)

print(response_short12.choices[0].message.content)
print(f"\nInput tokens: {response_short12.usage.prompt_tokens}, Output tokens: {response_short12.usage.completion_tokens}")

print("Within 10-token target:", response_short12.usage.completion_tokens <= 10)

You watched one romcom recently

Input tokens: 40, Output tokens: 9
Within 10-token target: True


**Original:** 172 output tokens, gave a list of 5 reasons with explanations and offered more help at the end.

**Shortened:** 9 output tokens, answer was "You watched one romcom recently".

**Conclusion:** got under the 10 token target, went from 172 to 9 output tokens. The short answer is still meaningful, it is the same as the first reason from the long answer. Asking for max 6 words worked better than asking for 10 tokens because it can't count tokens.

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [126]:
response_13_1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain the importance of exercise in exactly one sentence."}],
    max_completion_tokens=800,
)

print(response_13_1.choices[0].message.content)

Exercise is important because it strengthens the body and mind, improves health, and helps prevent disease.


In [127]:
response_13_2 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain the importance of exercise in a single paragraph of 4-6 sentences. Do not use headings or lists."}],
    max_completion_tokens=800,
)

print(response_13_2.choices[0].message.content)

Exercise is important because it strengthens the heart, muscles, and bones while improving overall physical fitness. It also helps maintain a healthy weight, boosts energy levels, and lowers the risk of many chronic diseases such as diabetes and high blood pressure. Beyond physical benefits, exercise supports mental health by reducing stress, anxiety, and symptoms of depression. Regular movement can also improve sleep, mood, focus, and quality of life, making it a key part of a healthy routine.


In [128]:
response_13_3 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain the importance of exercise as a bullet-point list of 5 benefits. Each bullet should start with the benefit in bold, followed by a one-sentence explanation."}],
    max_completion_tokens=800,
)

print(response_13_3.choices[0].message.content)

- **Improves heart health** — Regular exercise strengthens the heart and helps reduce the risk of cardiovascular disease.
- **Boosts mental health** — Physical activity can lower stress, anxiety, and symptoms of depression while improving mood.
- **Supports weight management** — Exercise helps burn calories and maintain a healthy body weight.
- **Increases strength and endurance** — It builds muscle, improves stamina, and makes daily activities easier.
- **Enhances overall well-being** — Staying active can improve sleep, energy levels, and long-term quality of life.


#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [129]:
response_14_1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain Artificial Intelligence using one extended metaphor. Compare AI to something from programming."}],
    max_completion_tokens=800,
)

print(response_14_1.choices[0].message.content)

Think of **Artificial Intelligence as a very advanced sort of library system** inside a giant software city.

In a normal program, you often write a **specific book for every situation**. If the traffic light is red, do this. If the user clicks here, do that. If the file is missing, show this message. The programmer is like an author writing exact instructions for every chapter.

AI is different. AI is more like a **library with a clever head librarian** trained on millions of books. Instead of being told every single answer in advance, it has studied patterns from all those books and learned how to **find, combine, and predict useful information** when a new request comes in.

When you ask the AI a question, it’s like walking into that library and saying, “I need help writing a letter,” or “What does this code error mean?” The AI doesn’t pull out one fixed, prewritten page. Instead, it searches its learned experience, recognizes the shape of your request, and then **builds a response 

In [130]:
response_14_2 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain Artificial Intelligence through a short story with a main character who discovers what AI is and how it learns. Story should have a plotwist and the end"}],
    max_completion_tokens=800,
)

print(response_14_2.choices[0].message.content)

Mina first noticed the robot in the library because it was the only thing there that blinked.

It sat beside the return desk, a silver box on tiny wheels with a cheerful screen for a face. Every few seconds, it looked around as if it were trying to remember why it existed.

“Hello,” Mina said.

The screen lit up. “Hello, Mina.”

She froze. “You know my name?”

“You checked out *The Last Starship* three times,” the robot said. “And you always sit by the window.”

Mina smiled despite herself. “So you’re a robot.”

“I am an AI assistant,” it said.

“A what?”

The librarian, Mrs. Vale, looked up from her papers. “Artificial Intelligence,” she said. “That’s what it stands for.”

Mina repeated the words quietly. Artificial Intelligence. It sounded like a spell.

“Does that mean it thinks?” she asked.

“Sort of,” Mrs. Vale said. “It learns patterns from data.”

Mina frowned. “Like school?”

“Kind of,” the robot said. “Except instead of a teacher showing me answers one by one, I study many exa

In [131]:
response_14_3 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain Artificial Intelligence in a funny way, like a standup comedy bit."}],
    max_completion_tokens=800,
)

print(response_14_3.choices[0].message.content)

Sure — here’s a standup-comedy-style explanation of Artificial Intelligence:

---

Artificial Intelligence is basically when we taught a computer to “think,” and it immediately became that one friend who’s technically helpful but also weirdly confident.

You know how humans learn by experience? AI learns by being fed millions of examples until it goes,  
“Ah yes, now I understand. That is a cat. That is also a cat. That is a sofa, but honestly, I’m giving it 40% cat.”

That’s AI: a machine with the energy of a toddler who has seen the same flashcard 8 million times.

And people say, “AI is intelligent.”
No. AI is *pattern-recognizing with commitment*.

It doesn’t “know” things the way we know things.  
You know a chair is for sitting. AI sees a chair and says, “Statistically, this object appears in homes, offices, and the occasional bad back.”

AI is also incredibly eager to help, which is adorable and dangerous.  
Ask it to write an email, and it goes:  
“Certainly, here is a polite, 

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [132]:
response_15_1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain climate change to kids. Use short sentences, simple words."}],
    max_completion_tokens=800,
)

print(response_15_1.choices[0].message.content)

Climate change means the Earth is getting warmer.

This happens because people burn coal, oil, and gas.  
That puts gases in the air.  
These gases trap heat from the Sun.

A warmer Earth can cause:
- hotter days
- stronger storms
- less snow in some places
- dry land and droughts
- animals losing their homes

We can help by:
- saving energy
- riding bikes or walking
- planting trees
- using less plastic
- recycling

Small actions can help a lot.


In [133]:
response_15_2 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain climate change to the general public. Use medium difficulity language, say what they shouldn't do."}],
    max_completion_tokens=800,
)

print(response_15_2.choices[0].message.content)

Climate change means the long-term change in Earth’s average weather patterns, especially warming. The main reason is that people are releasing too many greenhouse gases, like carbon dioxide and methane, into the air. These gases trap heat, which makes the planet warmer.

### Why it matters
Climate change can cause:
- hotter temperatures and stronger heatwaves
- more extreme storms, floods, and droughts
- melting ice and rising sea levels
- damage to crops, forests, animals, and human health

### What causes it
The biggest human causes are:
- burning coal, oil, and gas for electricity, transport, and heating
- cutting down forests, which reduces the number of trees that absorb carbon dioxide
- farming and waste that release methane and other gases

### What people should do
- save energy at home
- use public transport, walk, bike, or carpool when possible
- reduce food waste
- recycle and reuse
- support clean energy and climate-friendly policies
- plant and protect trees

### What the

In [134]:
response_15_3 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Explain climate change to politicians. Use complex sentences, talk about nothing and how they could improve their ratings if they find solution"}],
    max_completion_tokens=800,
)

print(response_15_3.choices[0].message.content)

Climate change is, in essence, a long-term shift in weather patterns driven largely by human activity, and while it can be described in scientific terms, it is also, for anyone in public office, a matter of governance, risk, and public trust; if you treat it as an abstraction, it will remain one, but if you treat it as a problem affecting food prices, insurance costs, infrastructure, public health, and national security, it becomes much harder to ignore.

For politicians, the real issue is not only whether the climate is changing, because that debate is largely settled, but whether they are willing to respond with policies that are credible enough to reduce harm and practical enough to survive scrutiny, since voters may not reward complexity, yet they do tend to reward results, especially when those results are visible in lower bills, safer communities, fewer disasters, and a sense that someone, finally, is in charge.

If you are looking at this through the lens of public approval, the

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [135]:
response_16_1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Describe a sunset over the sea. Talk about its colours how they change sky, temperature. Write about 120 words."}],
    max_completion_tokens=800,
)

print(response_16_1.choices[0].message.content)

The sun sinks slowly toward the horizon, and the sea begins to glow with its fading light. At first, the sky is bright gold and pale orange, shimmering like a soft flame. Soon the colours deepen into richer shades of pink, coral, and warm red, spreading across the clouds in long streaks. Higher up, the blue of the day turns to violet and then to a cool, dusky grey. The water reflects everything, flickering with gold near the sun and darkening into silver and blue farther away. As the light fades, the air grows cooler and gentler, and the warmth of the day slips away. A quiet, calm feeling settles over the sea.


In [136]:
response_16_2 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Describe a sunse. Strict rule: do not use the words 'sun' or 'light' anywhere, including inside other words such as 'sunset', 'sunlight', 'daylight etc."}],
    max_completion_tokens=800,
)

print(response_16_2.choices[0].message.content)

text = response_16_2.choices[0].message.content.lower()
print("\nContains 'sun':", "sun" in text, "| Contains 'light':", "light" in text)

The sky softens into bands of amber, rose, and violet as the horizon slowly glows. Trees and rooftops become dark silhouettes, and long shadows fade into the fading colors. The air feels calmer, cooler, and still, as if the world is pausing between day and night.

Contains 'sun': False | Contains 'light': False


In [137]:
response_16_3 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Describe a sunset in the style of William Shakespeare, write a sonnet."}],
    max_completion_tokens=800,
)

print(response_16_3.choices[0].message.content)

When sinking Sol with ruby hand doth spill  
His molten fire on sea and field below,  
The west is clad in robes of roseate still,  
And cloud on cloud doth burn with tender glow.  

The mountains, dark as thought, keep watch in awe,  
Whilst day, like some proud king, to night doth yield;  
The birds, made soft by evening’s kindly law,  
Seek hush’d repose across the shadow’d field.  

Then breathes the air a balm of amber sweet,  
And every stream reflects that golden fire;  
The slow, brief hush wherein the hours meet  
Doth stir the heart with wonder and desire.  

O fairest end, wherein the day doth die,  
Yet leaves its light to linger in the sky.


#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

**Format:** just adding "exactly one sentence", "one paragraph" or "bullet points" changed the format exactly as asked. The content stayed almost the same, only the length and structure changed.

**Creativity:** the metaphor compared AI to a library with a clever librarian, and normal programs to a book written for every situation. The story was long with characters, explained how AI learns from examples and had a plot twist. The funny one was like a standup bit with jokes.

**Audience:** kids version had super short sentences and simple words. General public had headings and lists with causes and effects. Politicians version had long complex sentences focused on voters, ratings, jobs.

**Sunset:** adding details made the model talk exactly about colours, how the sky changes and temperature. Forbidding "sun" and "light" worked, the check showed neither word was used. The Shakespeare one wrote a real sonnet with old school words.

**Overall:** the more specific the prompt the more the response follows it.

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [138]:
PROMPT18 = """You are senior python developer. Write a Python function that returns the factorial of a number.
Requirements:
1.Use an iterative approach.
2.factorial(0) must return 1.
3.Do error handling, continue if error print out error.
Return ONLY the code with no explanation. MAKE NO MISTAKES"""

response_18 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT18}],
    max_completion_tokens=1000,
)

print(response_18.choices[0].message.content)

generated_code = response_18.choices[0].message.content.strip()

if generated_code.startswith("```"):
    generated_code = generated_code.split("\n", 1)[1].rsplit("```", 1)[0]

def factorial(n):
    try:
        if not isinstance(n, int):
            raise TypeError("Input must be an integer")
        if n < 0:
            raise ValueError("Input must be non-negative")
        result = 1
        for i in range(2, n + 1):
            result *= i
        return result
    except Exception as e:
        print(e)
        return None


In [139]:
def factorial(n):
    try:
        if n == 0:
            return 1
        if n < 0:
            raise ValueError("Factorial is not defined for negative numbers")
        result = 1
        for i in range(1, n + 1):
            result *= i
        return result
    except Exception as e:
        print(e)
        return None

for i in range(-5, 10):
    print(factorial(i))

Factorial is not defined for negative numbers
None
Factorial is not defined for negative numbers
None
Factorial is not defined for negative numbers
None
Factorial is not defined for negative numbers
None
Factorial is not defined for negative numbers
None
1
1
2
6
24
120
720
5040
40320
362880


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [140]:
BUGGY = """def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)"""

PROMPT19 = f"""You are proffesional debugger. Fix the syntax errors in this Python code. Return ONLY the corrected code,
with no explanation. MAKE NO MISTAKES.

{BUGGY}"""

response_19 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT19}],
    max_completion_tokens=1000,
)

print(response_19.choices[0].message.content)

fixed_code = response_19.choices[0].message.content.strip()
if fixed_code.startswith("```"):
    fixed_code = fixed_code.split("\n", 1)[1].rsplit("```", 1)[0]

def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))


In [141]:
def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))

15


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [142]:
TEXT20 = """Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."""

response_20 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": f"Summarize the following text in 2-3 sentences. Keep key facts.\n\n{TEXT20}"}],
    max_completion_tokens=800,
)

print(response_20.choices[0].message.content)

Generative AI (gen AI) uses deep learning models to create original content such as text, images, video, and audio from a user prompt by learning a simplified representation of training data and generating similar but not identical outputs. It has evolved from earlier statistical generative models into advanced deep learning approaches like VAEs, diffusion models, and transformers, with transformers powering many major tools such as ChatGPT, GPT-4, Copilot, BERT, Bard, and Midjourney.


#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [143]:
IDIOMS = """1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise." """


response_21_basic = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role": "user", "content": f"Translate these idioms into Lithuanian:\n{IDIOMS}"}],
)

print(response_21_basic.choices[0].message.content)

print("\n" + "=" * 60 + "\n")

PROMPT21 = f"""Translate each English idiom below into Lithuanian.
Do NOT translate word-for-word. Give the natural Lithuanian idiom that would have actually the same meaning.
- If no similar idioms exists, give the most natural phrase and say so.
{IDIOMS}"""

response_21_improved = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": "You are a native Lithuanian speaker with deep understanding of English language."},
        {"role": "user", "content": PROMPT21},
    ],
)

print(response_21_improved.choices[0].message.content)

1. **“It’s raining cats and dogs.”** — **Lyja kaip iš kibiro.**  
2. **“Break a leg.”** — **Sėkmės!** / **Nei plauko, nei pūko!**  
3. **“Kill two birds with one stone.”** — **Vienu šūviu nušauti du zuikius.**  
4. **“Spill the beans.”** — **Išplepėti paslaptį.**  
5. **“A blessing in disguise.”** — **Nėra to blogo, kas neišeitų į gera.**


1. **„Lija kaip iš kibiro.“**  
2. **„Sėkmės!“** – tiesioginio lietuviško idiomos atitikmens nėra; tai natūraliausias posakis, vartojamas linkint sėkmės.  
3. **„Vienu šūviu nušauti du zuikius.“**  
4. **„Išplepėti paslaptį.“** – tikslaus idiominio atitikmens nėra; tai natūraliausias posakis.  
5. **„Nėra to blogo, kas neišeitų į gera.“**


#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [144]:
PROMPT22 = """Write a story about a cat in exactly THREE SENTENCES.
DO NOT USE IN THE STORY LETTER E (no upper case nor lower case), before writing a word check if it has letter e if so change it.
Output only the three sentences, nothing else."""

response_22 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT22}],
    max_completion_tokens=2000,
)

print(response_22.choices[0].message.content)

story = response_22.choices[0].message.content
bad_words = [w for w in story.split() if "e" in w.lower()]
print("\nLetter 'e' count:", story.lower().count("e"))
print("Words with 'e':", bad_words or "none")

A sly cat sat on a mat, cool and calm in moonlight.  
It roamt a warm flat, softly latching on to a fat rat.  
At dawn, it napd, full and proud, with a purr that flot out in air.

Letter 'e' count: 0
Words with 'e': none


#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [145]:
import json

TRIP = ("Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the "
        "Park Hyatt before heading to Kyoto.")

# Method 1 - instructions in the prompt
PROMPT23 = f"""Extract the following from the text and return a JSON object with the fields
destination, start_date and stay_duration.

Text: "{TRIP}" """

response_23_prompt = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT23}],
)

print(response_23_prompt.choices[0].message.content)

try:
    print("\nParsed:", json.loads(response_23_prompt.choices[0].message.content))
except json.JSONDecodeError as e:
    print("\nCould not parse as JSON:", e)

{"destination":"Tokyo","start_date":"June 12th","stay_duration":"5 nights"}

Parsed: {'destination': 'Tokyo', 'start_date': 'June 12th', 'stay_duration': '5 nights'}


In [146]:
trip_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "trip_info",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "destination": {"type": "string", "description": "The first city of the trip"},
                "start_date": {"type": "string", "description": "Start date formatted as YYYY-MM-DD"},
                "stay_duration": {"type": "integer", "description": "Number of nights"},
            },
            "required": ["destination", "start_date", "stay_duration"],
            "additionalProperties": False,
        },
    },
}

response_23_schema = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": f"Extract the trip information from this text: {TRIP}"}],
    response_format=trip_schema,
)

print(response_23_schema.choices[0].message.content)

data = json.loads(response_23_schema.choices[0].message.content)
print("\nParsed:", data)
print("stay_duration is int:", isinstance(data["stay_duration"], int))

{"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5}

Parsed: {'destination': 'Tokyo', 'start_date': '2026-06-12', 'stay_duration': 5}
stay_duration is int: True


#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [147]:
PROMPT24 = """Create a Markdown table comparing 6 popular programming languages:
Python, Java, JavaScript.

Use exactly these columns: | Language | Use Case | Advantages | Disadvantages |
- Use Case: 1-2 main areas where the language is used.
- Advantages and Disadvantages: 2 short.
- Use only up to 6 short words.
Output ONLY the table, with no text before or after it."""

response_24 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT24}],
    max_completion_tokens=1500,
)

print(response_24.choices[0].message.content)

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Web; data science | Easy syntax; large library | Slower; high memory |
| Java | Enterprise; Android apps | Portable; strong ecosystem | Verbose; slower startup |
| JavaScript | Web front-end; Node.js | Runs in browsers; flexible | Inconsistent; callback complexity |


| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Web, data science | Easy syntax; huge libraries | Slower than compiled languages |
| Java | Enterprise, Android | Portable; strong performance | Verbose; heavier memory use |
| JavaScript | Web, servers | Runs in browsers; flexible | Weird quirks; async complexity |

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [148]:
PROMPT25 = """Create a 5-question multiple-choice quiz about machine learning basics.

Follow this EXACT format:
- First line: the title **Machine learning basics quiz** in bold, then a blank line.
- Each question is numbered "1. " then "2. " etc.
- Each question has exactly four options on separate lines, labelled "a)", "b)", "c)", "d)".
- Every option line except the last one ends with two spaces.
- Correct answer MUST have indicator (*)
- Leave one blank line between questions.
- Output only the quiz, with NO extra text.
"""

response_25 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT25}],
    max_completion_tokens=1800,
)

print(response_25.choices[0].message.content)

**Machine learning basics quiz**

1. What is machine learning?
a) A method for programming computers to learn from data (*)  
b) A way to store files efficiently  
c) A type of computer hardware  
d) A language for writing websites  

2. Which of the following is a common supervised learning task?
a) Clustering similar customers  
b) Predicting house prices (*)  
c) Reducing image size  
d) Compressing data  

3. What is a training dataset used for?
a) To evaluate the final model only  
b) To teach the model patterns from data (*)  
c) To replace the model after deployment  
d) To remove all errors from data  

4. What does overfitting mean?
a) The model performs well only on training data and poorly on new data (*)  
b) The model learns too slowly  
c) The dataset is too large  
d) The features are perfectly balanced  

5. Which metric is often used to evaluate classification models?
a) Accuracy (*)  
b) Mean squared error  
c) File size  
d) Learning rate


**Machine learning basics quiz**

1. What is the main goal of machine learning?  
a) To write code faster  
b) To make predictions or decisions from data (*)  
c) To replace all databases  
d) To store files efficiently  

2. Which of the following is a common type of machine learning?  
a) Supervised learning (*)  
b) Visual programming  
c) Manual labeling only  
d) Cloud hosting  

3. What is training data used for?  
a) To test a model after deployment  
b) To teach a model patterns from examples (*)  
c) To erase model errors  
d) To increase internet speed  

4. Which metric is often used to evaluate a classification model?  
a) Accuracy (*)  
b) File size  
c) RAM capacity  
d) Screen resolution  

5. What does overfitting mean?  
a) The model performs well on training data but poorly on new data (*)  
b) The model is too small to run  
c) The model uses no data  
d) The model always predicts randomly

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [ ]:
PROMPT_26 = """Create a complete Markdown course outline for beginners titled
"Mastering Prompt Engineering for AI Applications".

Follow EXACTLY this structure, section titles, heading levels, and separators:

## Course Outline: Mastering Prompt Engineering for AI Applications
---
### Course Overview
Write 3 to 4 sentences explaining what the course teaches, what it brings, and who it targets to.
#### Objectives
Provide exactly 3 bullet points describing the main learning goals.
#### Target Audience
Provide exactly 3 bullet points describing the intended learners.
---
### Module Breakdown
Create exactly 5 modules, progressing from basic concepts to a final project. Separate each module with ---.

Use this format for every module:

#### Module N: <Title>
- **Description:** 1-2 sentences describing the module.
- **Key Learning Outcomes:**
  1. <Outcome>
  2. <Outcome>
  3. <Outcome>
- **Interactive Assignment:** One practical, hands-on task.
---
### Assessment Plan
Create a Markdown table with these columns:
| Assessment Method | Description | Weightage (%) |

Include exactly 4 assessment methods with weights totaling 100%.
---
### Certification Criteria
Write one introductory sentence followed by exactly 3 numbered, measurable requirements (e.g., minimum scores).
---
### Marketing Strategy
Provide exactly 5 steps using this format:

#### Step N: <Title>
- Include 2 to 3 bullet points describing practical marketing actions.

You can introduce new ideas for module content and marketing, but you should keep everything focused on beginner-level prompt engineering.

Output ONLY the Markdown course outline."""

response_26 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": PROMPT_26}],
    max_completion_tokens=4000,
)

print(response_26.choices[0].message.content)

## Course Outline: Mastering Prompt Engineering for AI Applications
---
### Course Overview
This course introduces beginners to the fundamentals of prompt engineering and how to apply them effectively across modern AI applications. Learners will explore practical techniques for writing clear, structured, and goal-oriented prompts that produce better AI outputs. The course also covers common mistakes, iterative refinement, and real-world use cases to help students build confidence. It is designed for students, professionals, and creators who want to use AI more effectively in everyday tasks and projects.

#### Objectives
- Understand the core principles of prompt engineering and how AI responds to instructions.
- Learn how to design, test, and refine prompts for better accuracy and relevance.
- Apply beginner-friendly prompting techniques to practical AI use cases.

#### Target Audience
- Beginners with little or no prior experience in prompt engineering.
- Students and professionals wh

## Course Outline: Mastering Prompt Engineering for AI Applications
---
### Course Overview
This course introduces beginners to the fundamentals of prompt engineering and how to apply them effectively across modern AI applications. Learners will explore practical techniques for writing clear, structured, and goal-oriented prompts that produce better AI outputs. The course also covers common mistakes, iterative refinement, and real-world use cases to help students build confidence. It is designed for students, professionals, and creators who want to use AI more effectively in everyday tasks and projects.

#### Objectives
- Understand the core principles of prompt engineering and how AI responds to instructions.
- Learn how to design, test, and refine prompts for better accuracy and relevance.
- Apply beginner-friendly prompting techniques to practical AI use cases.

#### Target Audience
- Beginners with little or no prior experience in prompt engineering.
- Students and professionals who want to improve productivity using AI tools.
- Creators, entrepreneurs, and knowledge workers interested in AI-assisted workflows.

---
### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module explains what prompt engineering is and why it matters in AI applications. Learners will discover how prompt quality affects AI responses and outcomes.
- **Key Learning Outcomes:**
  1. Define prompt engineering in simple terms.
  2. Identify the role of prompts in guiding AI behavior.
  3. Recognize the difference between vague and effective prompts.
- **Interactive Assignment:** Write three prompts for the same task and compare the differences in AI responses.

---

#### Module 2: Building Clear and Effective Prompts
- **Description:** This module focuses on the structure of a strong prompt, including clarity, context, and task instructions. Learners will practice crafting prompts that are specific and easy for AI to follow.
- **Key Learning Outcomes:**
  1. Use clear instructions to improve AI output quality.
  2. Add relevant context to guide model responses.
  3. Structure prompts for consistent and usable results.
- **Interactive Assignment:** Rewrite five poorly written prompts into clear, effective versions.

---

#### Module 3: Common Prompting Techniques for Beginners
- **Description:** This module introduces practical prompting techniques such as examples, role assignment, and step-by-step instructions. Learners will use these methods to make prompts more reliable and focused.
- **Key Learning Outcomes:**
  1. Apply few-shot prompting using simple examples.
  2. Use role-based prompts to shape AI responses.
  3. Break complex tasks into smaller prompting steps.
- **Interactive Assignment:** Create a prompt that asks the AI to act as a helpful assistant and complete a simple task in steps.

---

#### Module 4: Testing, Refining, and Troubleshooting Prompts
- **Description:** This module teaches learners how to evaluate prompt performance and improve weak outputs through iteration. Learners will also learn to identify common issues such as ambiguity, missing context, and overly broad instructions.
- **Key Learning Outcomes:**
  1. Test prompts and analyze the quality of responses.
  2. Improve prompts through iterative refinement.
  3. Troubleshoot common prompt failures and correct them.
- **Interactive Assignment:** Take one prompt, test it three times, and refine it after each output to improve the result.

---

#### Module 5: Final Project and Real-World Applications
- **Description:** This module brings together the skills from the course in a beginner-friendly final project. Learners will design a small prompt workflow for a real-world use case such as content drafting, brainstorming, or customer support.
- **Key Learning Outcomes:**
  1. Combine multiple prompt engineering techniques in one project.
  2. Design prompts for a practical AI application.
  3. Present a simple prompting workflow with clear purpose and results.
- **Interactive Assignment:** Build and submit a mini prompt toolkit for one real-world scenario, including at least three prompts and their intended uses.

---
### Assessment Plan

| Assessment Method | Description | Weightage (%) |
|---|---|---:|
| Quizzes | Short quizzes at the end of modules to check understanding of core concepts. | 20 |
| Practical Prompt Exercises | Hands-on prompt writing and refinement activities completed throughout the course. | 30 |
| Module Assignments | Graded tasks that demonstrate the ability to apply prompting techniques in context. | 25 |
| Final Project | A beginner-level prompt workflow or toolkit showing practical use of learned skills. | 25 |

---
### Certification Criteria
To earn certification, learners must demonstrate both conceptual understanding and practical prompt-writing ability.
1. Achieve a minimum overall score of 70%.
2. Complete at least 4 out of 5 module assignments.
3. Submit the final project with a score of at least 75%.

---
### Marketing Strategy

#### Step 1: Define the Beginner Value Proposition
- Highlight that the course requires no prior AI or coding experience.
- Emphasize practical benefits such as saving time and improving AI output quality.
- Use simple messaging focused on everyday AI use cases.

#### Step 2: Create Sample Prompt Demonstrations
- Share before-and-after examples showing weak prompts versus improved prompts.
- Publish short demo posts or videos on social platforms and learning communities.
- Show how prompt refinement changes results in a visible, relatable way.

#### Step 3: Build Trust with Free Learning Assets
- Offer a free prompt cheat sheet or mini guide as a lead magnet.
- Provide a short sample lesson to let learners experience the teaching style.
- Include testimonials or example outcomes from pilot learners if available.

#### Step 4: Promote Through Career and Productivity Channels
- Target audiences interested in productivity, content creation, and AI tools.
- Share the course in newsletters, online communities, and professional networks.
- Position the course as useful for students, freelancers, and office workers.

#### Step 5: Encourage Enrollment with Limited-Time Offers
- Use early-bird pricing or bonus resources for first-time enrollments.
- Offer bundle deals with related AI or productivity courses.
- Create a clear call to action that encourages immediate registration.

####